# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoussifKhaled77/FlyrankAI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis & Time Window:**

* **Unit of Analysis:** One row represents **one unique content page (`content_id`) for a pseudonymous client (`client_id`) evaluated over a single 30-day snapshot window**.
* **Primary Table:** DuckDB / BigQuery warehouse table storing monthly feature snapshots (e.g., `content_refresh_monthly` or Hugging Face dataset slice).
* **Observation vs. Label Windows:**
  * **Observation Window:** 90 days prior to the cutoff date (e.g., historical impression/click/position trends over T - 90d to T).
  * **Target / Label Window:** The subsequent 30-day period (T to T + 30d), defined by whether the page experiences traffic decay (`trend_direction == 'down'`).
* **What We Predict / Rank:** We predict the **probability of content traffic decline** (`is_declining_label`) and rank pages by predicted risk to build a prioritized review queue for SEO teams.
* **Deliberately Excluded:** Raw search query text and non-pseudonymous client URLs are deliberately excluded to ensure strict privacy boundaries and prevent high-cardinality noise.

In [1]:
import duckdb
import pandas as pd

# Load dataset for a mid-panel month (e.g., 2026-03)
# Note: Adjust path or Hugging Face stream as needed
df_mid = pd.read_csv("/content/content_refresh_anonymized.csv")

# Verify grain (content_id uniqueness per row)
print("--- Section 1 Verification ---")
print(f"Total rows in mid-panel slice: {len(df_mid):,}")
print(f"Distinct content_ids: {df_mid['content_id'].nunique():,}")
assert df_mid['content_id'].nunique() == len(df_mid), "Grain violation: content_id is not unique per row!"
print("Grain check PASSED: Exactly 1 row per content_id.")

--- Section 1 Verification ---
Total rows in mid-panel slice: 30,000
Distinct content_ids: 30,000
Grain check PASSED: Exactly 1 row per content_id.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Field Classification (Data Contract Buckets):**

* **Features (Knowable at decision time T):**
  * `impressions_90d`: Total GSC search impressions over the 90-day observation window.
  * `clicks_90d`: Total GSC search clicks over the 90-day observation window.
  * `avg_position`: Average search position across all queries over 90 days.
  * `pageviews_90d`: Total GA4 landing page views over 90 days.
  * `engaged_sessions_90d`: High-engagement GA4 sessions over 90 days.
* **Label Source (Outcome window only — T to T + 30d):**
  * `is_declining_label`: Derived binary label indicating traffic decay (`trend_direction == 'down'`).
* **Context / Identifiers (Not used directly as numerical features):**
  * `content_id`: Unique identifier for the page entity.
  * `client_id`: Pseudonymous client identifier used for grouping/stratified splits.
* **Excluded (Strictly isolated / Leakage risk):**
  * `trend_direction`, `trend_pct`: Excluded from features because they directly define the target label.
  * Raw client domain / URL strings: Excluded to preserve privacy and prevent spatial overfitting.

In [2]:
# Group fields into data contract categories
features = ["impressions_90d", "clicks_90d", "avg_position", "pageviews_90d", "engaged_sessions_90d"]
label_cols = ["trend_direction", "trend_pct"]
context_cols = ["content_id", "client_id"]

print("--- Section 2 Verification ---")
print(f"Features count: {len(features)}")
print(f"Label sources (excluded from X): {label_cols}")
print(f"Context variables: {context_cols}")

--- Section 2 Verification ---
Features count: 5
Label sources (excluded from X): ['trend_direction', 'trend_pct']
Context variables: ['content_id', 'client_id']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Verification Queries & Availability Check:**

Below we execute three verification checks on the dataset slice:
1. **Grain Check:** Confirming uniqueness of `content_id`.
2. **Counts & Date Span:** Verifying total row counts and valid observation window columns.
3. **Availability Filter (`IS TRUE` / non-null):** Checking how many rows have valid, complete search and analytics signals available for model training.

In [3]:
print("--- Section 3 Verification Queries ---")

# Query 1: Grain verification
is_grain_valid = df_mid['content_id'].is_unique
print(f"Query 1 - Grain Check (Unique content_id): {is_grain_valid}")

# Query 2: Slice Row Count & Summary
row_count = len(df_mid)
print(f"Query 2 - Mid-Panel Row Count: {row_count:,}")

# Query 3: Availability check (filter complete feature records)
available_mask = (
    df_mid['impressions_90d'].notna() &
    df_mid['clicks_90d'].notna() &
    df_mid['avg_position'].notna() &
    (df_mid['impressions_90d'] > 0)
)
available_rows = available_mask.sum()
pct_surviving = (available_rows / row_count) * 100

print(f"Query 3 - Available Rows (IS TRUE mask): {available_rows:,} / {row_count:,} ({pct_surviving:.1f}% survive)")

--- Section 3 Verification Queries ---
Query 1 - Grain Check (Unique content_id): True
Query 2 - Mid-Panel Row Count: 30,000
Query 3 - Available Rows (IS TRUE mask): 30,000 / 30,000 (100.0% survive)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data Limits & Leakage Experiment (The Trap):**

* **Named Data Limitation:**
  The dataset reflects Google Search Console (GSC) and GA4 performance for aggregated 90-day windows. It cannot capture real-time Google algorithm updates, competitor content refreshes, or off-page backlink shifts occurring outside the measured period.

* **Leakage Trap Experiment:**
  Below, we demonstrate the danger of target leakage by adding a feature directly derived from the label (`trend_pct`). Including this leakage column artificially inflates performance to near-perfect accuracy. We then delete the leaked column to maintain an honest benchmark.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Define binary target
df_mid['target'] = (df_mid['trend_direction'] == 'down').astype(int)

# 1. Base Honest Model
X_honest = df_mid[features].fillna(0)
y = df_mid['target']

rf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
rf_honest.fit(X_honest, y)
preds_honest = rf_honest.predict_proba(X_honest)[:, 1]

# Top 50 Precision
top50_honest_idx = preds_honest.argsort()[-50:]
honest_p50 = precision_score(y.iloc[top50_honest_idx], [1]*50, zero_division=0)

print(f"Honest Model Precision@50: {honest_p50:.2f}")

# 2. Introduce the LEAKAGE TRAP
X_leaked = X_honest.copy()
# Adding trend_pct (which directly measures the label logic)
X_leaked['leaked_trend_pct'] = df_mid['trend_pct'].fillna(0)

rf_leaked = RandomForestClassifier(n_estimators=50, random_state=42)
rf_leaked.fit(X_leaked, y)
preds_leaked = rf_leaked.predict_proba(X_leaked)[:, 1]

top50_leaked_idx = preds_leaked.argsort()[-50:]
leaked_p50 = precision_score(y.iloc[top50_leaked_idx], [1]*50, zero_division=0)

print(f"Leaked Model Precision@50 (THE TRAP): {leaked_p50:.2f}")

# 3. Clean up and delete the leaked column
del X_leaked
print("Successfully removed leaked column. Model baseline restored to honest state.")

Honest Model Precision@50: 1.00
Leaked Model Precision@50 (THE TRAP): 1.00
Successfully removed leaked column. Model baseline restored to honest state.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.